<a href="https://colab.research.google.com/github/Eugene1617/student-warefare/blob/main/studentapi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import sqlite3
from datetime import datetime
import joblib
import uuid

app = FastAPI(title="Abuse Detection API", version="2.0.0")

model = joblib.load("/content/drive/MyDrive/second year datascience project materils/student_welfare_model3.pkl")

conn = sqlite3.connect("reports.db", check_same_thread=False)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS reports (
    reportid TEXT PRIMARY KEY,
    text TEXT,
    date TEXT,
    time TEXT,
    label TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS alerts (
    alertid TEXT PRIMARY KEY,
    reportid TEXT,
    level TEXT,
    text TEXT,
    time TEXT
)
""")

conn.commit()

class TextInput(BaseModel):
    text: str

HIGH_RISK = {"critical", "high"}

def trigger_alert(reportid, label, text):
    if label in HIGH_RISK:
        alertid = str(uuid.uuid4())

        cursor.execute("""
        INSERT INTO alerts (alertid, reportid, level, text, time)
        VALUES (?, ?, ?, ?, ?)
        """, (
            alertid,
            reportid,
            label,
            text,
            datetime.now().isoformat()
        ))

        conn.commit()

@app.get("/")
def home():
    return {"message": "Abuse Detection API running"}

@app.get("/health")
def health():
    return {"status": "ok"}

@app.get("/reports")
def get_reports():
    cursor.execute("SELECT * FROM reports")
    rows = cursor.fetchall()

    return {
        "count": len(rows),
        "data": rows
    }

@app.get("/alerts")
def get_alerts():
    cursor.execute("SELECT * FROM alerts")
    rows = cursor.fetchall()

    return {
        "count": len(rows),
        "data": rows
    }

@app.post("/predict")
def predict(data: TextInput):

    text = data.text.strip()

    if not text:
        raise HTTPException(status_code=400, detail="Text cannot be empty")

    try:
        label = model.predict([text])[0]

        now = datetime.now()
        date = now.strftime("%Y-%m-%d")
        time = now.strftime("%H:%M:%S")

        reportid = str(uuid.uuid4())

        cursor.execute("""
        INSERT INTO reports (reportid, text, date, time, label)
        VALUES (?, ?, ?, ?, ?)
        """, (reportid, text, date, time, label))

        conn.commit()

        trigger_alert(reportid, label, text)

        return {
            "status": "submitted",
            "reportid": reportid,
            "date": date,
            "time": time,
            "alert": label in HIGH_RISK
        }

    except Exception as e:
        raise HTTPException(status_code=500, detail="Submission failed")

@app.delete("/reports")
def clear_reports():
    cursor.execute("DELETE FROM reports")
    conn.commit()

    return {"message": "all reports deleted"}

In [ ]:
from fastapi.testclient import TestClient
client = TestClient(app)

dataa=client.get("/reports")
print(dataa.json())

{'count': 3, 'data': [['b9ac4ef8-87dd-42a1-ba15-7c237e4b25ff', 'i want to kill my self', '2026-06-18', '16:33:27', 'critical'], ['9dee6eda-dcfc-41df-b906-19144d370e4c', 'i want to kill my self', '2026-06-18', '16:33:40', 'critical'], ['3a0160c8-3559-40ef-9b85-9840f77f6e50', 'i want to kill my self', '2026-06-18', '16:34:15', 'critical']]}
